In [0]:
/*
Project: Adventure Works Analytics
Purpose: Referential Integrity Analysis
Platform: Databricks SQL
Catalog: fea_academy
Schema: adventure_works
*/

-- =====================================================
-- Country Region to State Province
-- =====================================================

/*
Insights:
    - Are there any states/provinces with no corresponding country?
    - Does Namibia appear among the records with no match?
    - Are there any other inconsistent codes?
*/

SELECT
    sp.stateprovinceid,
    sp.name AS stateprovince_name,
    sp.countryregioncode,
    cr.name AS country_name,
    CASE
        WHEN sp.countryregioncode IS NULL
            THEN 'NULL COUNTRY CODE'
        WHEN cr.countryregioncode IS NULL
            THEN 'UNMATCHED COUNTRY CODE'
        ELSE 'MATCHED'
    END AS relationship_status
FROM `fea_academy`.`adventure_works`.`person_stateprovince` sp
LEFT JOIN `fea_academy`.`adventure_works`.`person_countryregion` cr
    ON sp.countryregioncode = cr.countryregioncode
WHERE cr.countryregioncode IS NULL
ORDER BY sp.name;

/* 
Notes:
    - The record for Namibia with `countryregioncode = NULL` did not cause an inconsistency in this query, as there appear to be no states/provinces associated with this null code
    - Therefore, we are keeping the proposed correction—mapping `NULL` to `NA`—within the transformation layer, though it does not affect the geographic relationship currently under analysis
*/

-- =====================================================
-- Customer to Person
-- =====================================================

/*
Insights:
    - How many customers are associated with individuals?
    - How many customers are associated with stores?
    - Whether there are any customers with a `personid` that has no corresponding entry in the `person_person` table
*/

SELECT
    COUNT(*) AS total_customers,
    COUNT(c.personid) AS customers_with_person,
    COUNT(c.storeid) AS customers_with_store,
    SUM(
        CASE
            WHEN c.personid IS NOT NULL
             AND p.businessentityid IS NULL
            THEN 1 ELSE 0
        END
    ) AS unmatched_persons
FROM `fea_academy`.`adventure_works`.`sales_customer` c
LEFT JOIN `fea_academy`.`adventure_works`.`person_person` p
    ON CAST(c.personid AS BIGINT) = p.businessentityid;

/*
Notes:
    - The sum of customers with a `personid` and customers with a `storeid` is 20,455, which is higher than the total of 19,820
    - This occurs because the two categories are not necessarily mutually exclusive. A customer can have both a `personid` and a `storeid` simultaneously
*/

-- =====================================================
-- Customer Type Distribution
-- =====================================================

SELECT
    CASE
        WHEN personid IS NOT NULL AND storeid IS NOT NULL
            THEN 'PERSON AND STORE'
        WHEN personid IS NOT NULL AND storeid IS NULL
            THEN 'PERSON ONLY'
        WHEN personid IS NULL AND storeid IS NOT NULL
            THEN 'STORE ONLY'
        ELSE 'WITHOUT PERSON OR STORE'
    END AS customer_type,
    COUNT(*) AS total_customers
FROM `fea_academy`.`adventure_works`.`sales_customer`
GROUP BY 1
ORDER BY total_customers DESC;

/*
Notes:
    - Most customers have an association with a person
    - There are 701 customers associated exclusively with stores
    - The 635 customers associated with both a person and a store should not be arbitrarily classified into only one category
    - We found no customers without an association to either a person or a store
FINDINGS:
    - Total customers: 19,820
    - PERSON ONLY: 18,484 (93.26%)
    - STORE ONLY: 701 (3.54%)
    - PERSON AND STORE: 635 (3.20%)
    - No customers without personid or storeid were identified
    - All non-null personid values matched person_person.businessentityid
MODELING CONSIDERATIONS:
    - Customer classification must consider both personid and storeid
    - PERSON AND STORE records must not be arbitrarily assigned to a single category
    - Preserve both relationships in the staging layer
*/

-- =====================================================
-- Order Details to Products and Orders
-- =====================================================

SELECT
    COUNT(*) AS total_order_details,
    SUM(CASE WHEN h.salesorderid IS NULL THEN 1 ELSE 0 END)
        AS unmatched_orders,
    SUM(CASE WHEN p.productid IS NULL THEN 1 ELSE 0 END)
        AS unmatched_products
FROM `fea_academy`.`adventure_works`.`sales_salesorderdetail` d
LEFT JOIN `fea_academy`.`adventure_works`.`sales_salesorderheader` h
    ON d.salesorderid = h.salesorderid
LEFT JOIN `fea_academy`.`adventure_works`.`production_product` p
    ON d.productid = p.productid;

/*
Notes:
    - All order details have a matching sales order header
    - All order details have a matching product
    - Both relationships passed the referential integrity check
*/

-- =====================================================
-- Orders to Customers
-- =====================================================

SELECT
    COUNT(*) AS total_orders,
    SUM(
        CASE
            WHEN c.customerid IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_customers
FROM `fea_academy`.`adventure_works`.`sales_salesorderheader` h
LEFT JOIN `fea_academy`.`adventure_works`.`sales_customer` c
    ON h.customerid = c.customerid;

/*
Notes:
    - All sales orders have a matching customer record
    - The referential integrity check passed
*/

-- =====================================================
-- Orders to Addresses
-- =====================================================

/*
Insights:
    - billtoaddressid -> person_address.addressid
    - shiptoaddressid -> person_address.addressid
*/

SELECT
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN bill_address.addressid IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_billto_addresses,

    SUM(
        CASE
            WHEN ship_address.addressid IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_shipto_addresses

FROM `fea_academy`.`adventure_works`.`sales_salesorderheader` h

LEFT JOIN `fea_academy`.`adventure_works`.`person_address` bill_address
    ON h.billtoaddressid = bill_address.addressid

LEFT JOIN `fea_academy`.`adventure_works`.`person_address` ship_address
    ON h.shiptoaddressid = ship_address.addressid;

/*
Notes:
RESULT:
    - All sales orders have matching billing addresses
    - All sales orders have matching shipping addresses
    - Both relationships passed the referential integrity check
MODELING CONSIDERATION:
    - Preserve billing and shipping locations as separate roles when defining geographic dimensions

Insights:
    - The initial recommendation is to use the delivery address (shiptoaddressid) as the primary geographic reference, as it represents the physical destination of the order
*/

-- =====================================================
-- Orders to Credit Cards
-- =====================================================

SELECT
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN h.creditcardid IS NULL THEN 1
            ELSE 0
        END
    ) AS orders_without_credit_card,

    SUM(
        CASE
            WHEN h.creditcardid IS NOT NULL
             AND cc.creditcardid IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_credit_cards

FROM `fea_academy`.`adventure_works`.`sales_salesorderheader` h

LEFT JOIN `fea_academy`.`adventure_works`.`sales_creditcard` cc
    ON h.creditcardid = cc.creditcardid;

/*
Notes:
RESULT:
    - All non-null creditcardid values have a matching credit card record
    - No referential integrity issues were identified
MODELING CONSIDERATIONS:
    - Preserve orders without creditcardid
    - Handle missing card information as NULL or a designated "Not Applicable" category in the analytical layer
    - Do not classify missing credit card information as a data quality error
*/

-- =====================================================
-- Orders to Sales Reasons
-- =====================================================

SELECT
    COUNT(*) AS total_order_reason_records,

    SUM(
        CASE
            WHEN sr.salesreasonid IS NULL THEN 1
            ELSE 0
        END
    ) AS unmatched_sales_reasons

FROM `fea_academy`.`adventure_works`.`sales_salesorderheadersalesreason` hr

LEFT JOIN `fea_academy`.`adventure_works`.`sales_salesreason` sr
    ON hr.salesreasonid = sr.salesreasonid;

/*
Notes:
RESULT:
    - All salesreasonid values have a matching sales reason record
    - The referential integrity check passed
MODELING CONSIDERATION:
    - Preserve the many-to-many relationship between sales orders and sales reasons through a bridge table
    - Avoid duplicating sales metrics when joining sales reasons directly to sales order details
*/

-- =====================================================
-- Sales Order Status Distribution
-- =====================================================

/*
Insights:
    - What status codes exist in the database?
    - How many orders are associated with each status?
    - What the percentage share of each status is?
*/

SELECT
    status,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM `fea_academy`.`adventure_works`.`sales_salesorderheader`
GROUP BY status
ORDER BY status;

/*
Notes:
    - Status 5 -> Shipped
    - Total sales orders: 31,465
    - Status 5: 31,465 orders (100%)
    - No other status values were found
MODELING CONSIDERATIONS:
    - Preserve the source status code in the analytical layer
    - The status description should be validated before being documented
    - The dataset does not allow comparisons across different order statuses
*/

-- =====================================================
-- Sales Financial Metrics Overview
-- =====================================================

WITH detail_metrics AS (
    SELECT
        COUNT(*) AS total_order_details,
        SUM(orderqty) AS total_quantity,
        SUM(unitprice * orderqty) AS gross_sales,
        SUM(unitprice * orderqty * unitpricediscount) AS total_discounts,
        SUM(unitprice * orderqty * (1 - unitpricediscount)) AS net_sales
    FROM `fea_academy`.`adventure_works`.`sales_salesorderdetail`
),

header_metrics AS (
    SELECT
        COUNT(*) AS total_orders,
        MIN(orderdate) AS first_order_date,
        MAX(orderdate) AS last_order_date,
        SUM(subtotal) AS header_subtotal,
        SUM(taxamt) AS total_tax,
        SUM(freight) AS total_freight,
        SUM(totaldue) AS total_due
    FROM `fea_academy`.`adventure_works`.`sales_salesorderheader`
)

SELECT
    ROUND(h.total_orders, 2) AS total_orders,
    ROUND(d.total_order_details, 2) AS total_order_details,
    ROUND(d.total_quantity, 2) AS total_quantity,
    ROUND(d.gross_sales, 2) AS gross_sales,
    ROUND(d.total_discounts, 2) AS total_discounts,
    ROUND(d.net_sales, 2) AS net_sales,
    ROUND(h.header_subtotal, 2) AS header_subtotal,
    ROUND(h.total_tax, 2) AS total_tax,
    ROUND(h.total_freight, 2) AS total_freight,
    ROUND(h.total_due, 2) AS total_due,
    h.first_order_date,
    h.last_order_date
FROM detail_metrics d
CROSS JOIN header_metrics h;

/*
Notes:
    - The dataset contains 31,465 orders, comprising 121,317 items and 274,914 units, spanning the period from May 2011 to June 2014
    - The gross revenue, calculated based on the items (UnitPrice × OrderQty), is approximately US$ 110.37 million
    - Discounts total approximately US$ 527.51 thousand
    - The net value of the items (UnitPrice × OrderQty × (1 - UnitPriceDiscount)) is approximately US$ 109.85 million
    - The `header_subtotal` is essentially equal to the `net_sales`. The difference of approximately US$ 0.004 is likely due to numerical precision (floating-point arithmetic) rather than any commercial discrepancy
    - The `total_due` includes taxes and freight; therefore, it should not be confused with the net value of the items
*/

-- =====================================================
-- Annual Sales Financial Reconciliation
-- =====================================================

WITH detail_metrics AS (
    SELECT
        YEAR(TO_DATE(orderdate)) AS sales_year,
        SUM(d.unitprice * d.orderqty) AS gross_sales,
        SUM(
            d.unitprice * d.orderqty * d.unitpricediscount
        ) AS total_discounts,
        SUM(
            d.unitprice * d.orderqty *
            (1 - d.unitpricediscount)
        ) AS net_sales
    FROM `fea_academy`.`adventure_works`.`sales_salesorderdetail` d
    INNER JOIN `fea_academy`.`adventure_works`.`sales_salesorderheader` h
        ON d.salesorderid = h.salesorderid
    GROUP BY YEAR(TO_DATE(orderdate))
),

header_metrics AS (
    SELECT
        YEAR(TO_DATE(orderdate)) AS sales_year,
        COUNT(*) AS total_orders,
        SUM(subtotal) AS header_subtotal,
        SUM(taxamt) AS total_tax,
        SUM(freight) AS total_freight,
        SUM(totaldue) AS total_due
    FROM `fea_academy`.`adventure_works`.`sales_salesorderheader`
    GROUP BY YEAR(TO_DATE(orderdate))
)

SELECT
    h.sales_year,
    h.total_orders,
    ROUND(d.gross_sales, 2) AS gross_sales,
    ROUND(d.total_discounts, 2) AS total_discounts,
    ROUND(d.net_sales, 2) AS net_sales,
    ROUND(h.header_subtotal, 2) AS header_subtotal,
    ROUND(h.total_tax, 2) AS total_tax,
    ROUND(h.total_freight, 2) AS total_freight,
    ROUND(h.total_due, 2) AS total_due
FROM header_metrics h
INNER JOIN detail_metrics d
    ON h.sales_year = d.sales_year
ORDER BY h.sales_year;

/*
Notes:
    - Analysis period: May 2011 to June 2014
    - Total orders: 31,465
    - Gross sales in 2011: USD 12,646,112.16
    - The 2011 gross sales value exactly matches the required accounting reconciliation reference
FINANCIAL METRICS:
    - Gross Sales = SUM(UnitPrice * OrderQty)
    - Total Discounts = SUM(UnitPrice * OrderQty * UnitPriceDiscount)
    - Net Sales = SUM(UnitPrice * OrderQty * (1 - UnitPriceDiscount))
    - Header Subtotal represents the net value of order details
    - Total Due includes subtotal, tax, and freight
VALIDATION:
    - Annual gross sales reconciliation passed for 2011
    - Minor differences between detail-level net sales and header subtotal may result from decimal precision and rounding
    - 2014 contains data only through June 30 and is not a full year
MODELING CONSIDERATIONS:
    - Keep gross sales, discounts, net sales, tax, freight, and total due as separate analytical measures
    - Use net sales as the basis for discounted sales analysis
    - Preserve financial precision and apply consistent rounding rules
*/

-- =====================================================
-- Monthly Sales Performance
-- =====================================================

SELECT
    YEAR(TO_DATE(h.orderdate)) AS sales_year,
    MONTH(TO_DATE(h.orderdate)) AS sales_month,
    COUNT(DISTINCT h.salesorderid) AS total_orders,
    SUM(d.orderqty) AS total_quantity,
    ROUND(SUM(d.unitprice * d.orderqty), 2) AS gross_sales,
    ROUND(
        SUM(d.unitprice * d.orderqty * d.unitpricediscount), 2
    ) AS total_discounts,
    ROUND(
        SUM(d.unitprice * d.orderqty * (1 - d.unitpricediscount)), 2
    ) AS net_sales
FROM `fea_academy`.`adventure_works`.`sales_salesorderheader` h
INNER JOIN `fea_academy`.`adventure_works`.`sales_salesorderdetail` d
    ON h.salesorderid = d.salesorderid
GROUP BY
    YEAR(TO_DATE(h.orderdate)),
    MONTH(TO_DATE(h.orderdate))
ORDER BY
    sales_year,
    sales_month;

/*
Notes:
    - Analysis period: May 2011 to June 2014 (38 months)
    - July 2013: 1,740 orders
    - November 2013: 2,103 orders, the highest monthly order count
    - March 2014: 22,582 units sold, the highest monthly quantity
    - May 2014: USD 5,379,614.17 in gross sales, the highest monthly value
    - June 2014: USD 49,005.84 in gross sales, an unusually low value
DATA QUALITY INVESTIGATION:
    - June 2014 presents a significant decrease in gross sales
    - Investigate daily sales distribution and order-level values before classifying this variation as a business trend or anomaly
MODELING CONSIDERATIONS:
    - Keep order count, purchased quantity, gross sales, discounts, and net sales as separate measures
    - Use a proper date dimension for monthly and yearly analysis
*/

-- =====================================================
-- June 2014 Daily Sales Investigation
-- =====================================================

SELECT
    TO_DATE(h.orderdate) AS order_date,
    COUNT(DISTINCT h.salesorderid) AS total_orders,
    COUNT(d.salesorderdetailid) AS total_order_details,
    SUM(d.orderqty) AS total_quantity,
    ROUND(SUM(d.unitprice * d.orderqty), 2) AS gross_sales,
    ROUND(
        SUM(d.unitprice * d.orderqty * (1 - d.unitpricediscount)), 2
    ) AS net_sales
FROM `fea_academy`.`adventure_works`.`sales_salesorderheader` h
INNER JOIN `fea_academy`.`adventure_works`.`sales_salesorderdetail` d
    ON h.salesorderid = d.salesorderid
WHERE YEAR(TO_DATE(h.orderdate)) = 2014
  AND MONTH(TO_DATE(h.orderdate)) = 6
GROUP BY TO_DATE(h.orderdate)
ORDER BY order_date;

/*
Notes:
    - Orders were recorded on all 30 days of June 2014
    - Daily order count ranged from 19 to 51
    - Daily gross sales ranged from USD 949.64 to USD 2,643.61
    - The monthly sales decrease is not explained by missing sales days
CONCLUSION:
    - June 2014 presents an unusual reduction in sales value
    - Keep the records and flag this behavior for business validation
    - No data correction is justified based on this analysis alone
*/